# Phase 2 - GitHub Tool Explore

GitHub REST API로 LangChain 관련 저장소를 검색하고 `search_github_repos` 도구로 래핑합니다.

완료 기준:
- 검색 결과에 레포 이름, URL, stars 포함
- API 실패 시 raise 대신 문자열 반환
- 사용자 질문 1회당 `github_search` API 호출 최대 2회 제한

## 1. 환경 로드

In [ ]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "practice":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

github_token = os.getenv("GITHUB_TOKEN")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("GITHUB_TOKEN loaded:", bool(github_token))

## 2. GitHub API 직접 테스트

In [ ]:
import requests

from my_project.api_limits import create_query_limiter

query_limiter = create_query_limiter()
GITHUB_SEARCH_URL = "https://api.github.com/search/repositories"


def github_headers() -> dict[str, str]:
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        return {}
    return {"Authorization": f"Bearer {token}"}


def github_search_request(query: str, per_page: int = 10) -> requests.Response:
    params = {"q": f"langchain {query} in:name,description,readme", "sort": "stars", "per_page": per_page}
    return query_limiter.run(
        "github_search",
        lambda: requests.get(
            GITHUB_SEARCH_URL,
            headers=github_headers(),
            params=params,
            timeout=10,
        ),
    )


query_limiter.reset()
resp = github_search_request("rag", per_page=10)
print("status:", resp.status_code)
print("rate remaining:", resp.headers.get("X-RateLimit-Remaining"))
print("keys:", list(resp.json().keys())[:10])

## 3. 응답 파싱 확인

In [ ]:
items = resp.json().get("items", [])

for item in items[:5]:
    print(f"- {item['full_name']} (stars: {item['stargazers_count']})")
    print(f"  description: {item.get('description') or 'N/A'}")
    print(f"  url: {item['html_url']}")
    print()

## 4. Tool 래핑

In [ ]:
from langchain_core.tools import tool


RELEVANCE_KEYWORDS = {
    "langchain": 4,
    "langgraph": 4,
    "rag": 3,
    "retrieval": 3,
    "agent": 3,
    "llm": 2,
    "chatbot": 2,
    "vector": 2,
    "embedding": 2,
    "multimodal": 2,
    "orchestration": 2,
    "document": 1,
    "qa": 1,
}


def repo_relevance_score(item: dict, query: str) -> tuple[int, list[str]]:
    """레포 이름/설명/토픽 기반으로 프로젝트 관련도를 간단히 점수화합니다."""
    text = " ".join([
        item.get("full_name") or "",
        item.get("name") or "",
        item.get("description") or "",
        " ".join(item.get("topics") or []),
    ]).lower()
    query_terms = [term.lower() for term in query.split() if len(term) >= 3]

    matched = []
    score = 0
    for keyword, weight in RELEVANCE_KEYWORDS.items():
        if keyword in text:
            score += weight
            matched.append(keyword)

    for term in query_terms:
        if term in text and term not in matched:
            score += 2
            matched.append(term)

    stars = int(item.get("stargazers_count") or 0)
    score += min(stars // 1000, 5)
    return score, matched


def select_relevant_repos(items: list[dict], query: str, limit: int = 3, min_score: int = 4) -> list[tuple[dict, int, list[str]]]:
    scored = []
    for item in items:
        score, matched = repo_relevance_score(item, query)
        if score >= min_score:
            scored.append((item, score, matched))

    scored.sort(key=lambda row: (row[1], int(row[0].get("stargazers_count") or 0)), reverse=True)
    return scored[:limit]


def relevance_label(score: int) -> str:
    if score >= 11:
        return "높음"
    if score >= 6:
        return "중간"
    return "낮음"


def build_github_debug_adr(query: str, response: requests.Response, candidate_count: int, selected_count: int) -> str:
    """디버깅용 결정 기록. 숨은 사고 과정 대신 관찰 가능한 결정 근거만 남깁니다."""
    api_calls = query_limiter.snapshot().get("github_search", 0)
    request_url = response.url.replace(os.getenv("GITHUB_TOKEN") or "", "***")
    return "\n".join([
        "\n\n[Debug / ADR]",
        "ADR-Phase2-001: GitHub 저장소 검색 전략",
        f"- Context: 사용자 아이디어와 유사한 LangChain 구현 사례가 필요함",
        f"- Decision: GitHub Search API를 사용하고 검색어는 'langchain {query} in:name,description,readme'로 구성",
        f"- Rationale: GitHub stars 후보 10개를 먼저 가져온 뒤, LangChain/RAG/Agent 관련 키워드와 최소 점수로 재정렬",
        f"- API: github_search 호출 {api_calls}/2회",
        f"- Status: HTTP {response.status_code}, rate_limit_remaining={response.headers.get('X-RateLimit-Remaining')}",
        f"- Candidate count: {candidate_count}",
        f"- Selected count: {selected_count}",
        f"- Request URL: {request_url}",
    ])


@tool
def search_github_repos(query: str) -> str:
    """LangChain 관련 GitHub 레포지토리를 검색합니다.
    사용자 앱 아이디어와 유사한 구현 예시를 찾을 때 사용하세요.
    """
    try:
        response = github_search_request(query, per_page=10)
        response.raise_for_status()
        items = response.json().get("items", [])
        selected = select_relevant_repos(items, query, limit=3)
        debug_adr = build_github_debug_adr(query, response, len(items), len(selected))
        if not selected:
            return "관련 GitHub 레포지토리를 찾을 수 없습니다." + debug_adr

        results = []
        for item, score, matched in selected:
            results.append(
                f"- {item['full_name']} (stars: {item['stargazers_count']})\n"
                f"  설명: {item.get('description') or 'N/A'}\n"
                f"  URL: {item['html_url']}\n"
                f"  관련도: {relevance_label(score)} (score: {score})\n"
                f"  근거 키워드: {', '.join(matched) or 'N/A'}"
            )
        return "\n\n".join(results) + debug_adr
    except Exception as exc:
        return f"GitHub 검색 실패: {exc}"


def ask_github(query: str) -> str:
    """사용자 질문 1회 처리. limiter를 리셋해서 질문당 2회 제한 적용."""
    query_limiter.reset()
    return search_github_repos.invoke(query)


# 테스트
print(ask_github("agent skill"))